# Мост: многомерные вероятности → Flow Matching → контракты видеомодели

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../README.md).

**Связанные лекции:** [Lecture 1](https://diffusion.csail.mit.edu/2026/docs/20260120_Lecture_01.pdf) · [Lecture 2](https://diffusion.csail.mit.edu/2026/docs/20260122_Lecture_02.pdf) · [Lecture 3](https://diffusion.csail.mit.edu/2026/docs/20260123_Lecture_03.pdf) · [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Работайте в паре с [картой курса](../README.md). Здесь нет скачивания датасетов/весов и полной генерации.
Это маленькие проверяемые примеры на CPU PyTorch; они готовят к Lab 1–3, ComfyUI и чтению Hunyuan/Wan/Kandinsky.

Перед каждой ячейкой предскажите результат. После неё запишите, что проверка доказывает и чего не доказывает.
MLX — последний optional этап после корректности PyTorch; по умолчанию не импортируется.


In [ ]:
from pathlib import Path
import json, os
import numpy as np
import torch

def find_root():
    for p in (Path.cwd(),*Path.cwd().parents):
        if (p/'curriculum/references/materials_manifest.json').is_file():return p
    raise FileNotFoundError('Open this notebook from generative-ai-research.')
ROOT=find_root();OUT=ROOT/'curriculum/artifacts';OUT.mkdir(exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR',str(OUT/'mpl-cache'))
import matplotlib.pyplot as plt
torch.set_num_threads(4);torch.manual_seed(184)
rng=np.random.default_rng(184)
DEVICE='cpu'  # All default experiments are small and use no downloaded weights.
results={'scope':'small educational CPU experiments; no pretrained inference','mlx':'not_run'}
print('Device:',DEVICE,'Torch:',torch.__version__)


## 1. Один объект, batch и совместная вероятность

Одна случайная точка $X\in\mathbb R^d$ имеет covariance matrix $\Sigma$.
Batch `[B,d]` содержит B реализаций такого вектора. Независимость batch samples не означает независимость d координат.

Для Gaussian $s(x)=\nabla_x\log p(x)=-\Sigma^{-1}(x-\mu)$.
Проверим moments выборки и аналитический score через autograd.


In [ ]:
mu=torch.tensor([1.,-1.],dtype=torch.float64)
cov=torch.tensor([[1.,0.75],[0.75,2.]],dtype=torch.float64)
noise=torch.randn(10000,2,dtype=torch.float64)
samples=mu+noise@torch.linalg.cholesky(cov).T
empirical_cov=torch.cov(samples.T)
assert torch.max(torch.abs(empirical_cov-cov))<0.1
query=samples[:32].clone().requires_grad_(True)
distribution=torch.distributions.MultivariateNormal(mu,covariance_matrix=cov)
grad=torch.autograd.grad(distribution.log_prob(query).sum(),query)[0]
analytic=-(query-mu)@torch.linalg.inv(cov)
score_error=float((grad-analytic).abs().max().detach())
assert score_error<1e-10
print('Sample mean:',samples.mean(0));print('Sample covariance:',empirical_cov)
print('Score max error:',score_error)
results['gaussian_score_error']=score_error
plt.figure(figsize=(5,3));plt.scatter(*samples[:1500].T,s=3,alpha=.2)
plt.xlabel('x1');plt.ylabel('x2');plt.title('Коррелированные координаты одной Gaussian');plt.tight_layout();plt.show()


**Вопрос:** как изменится плотность при замене covariance на её diagonal?
Почему вероятность конкретного вектора равна нулю для continuous law, хотя density положительна?
Не интерпретируйте один pixel как независимую случайную величину без такого предположения модели.

## 2. Два time conventions описывают один путь

В MIT: $x_s=(1-s)\epsilon+s\,x_{data}$, velocity $x_{data}-\epsilon$, sampling $s:0\to1$.
В data-to-noise convention: $x_t=(1-t)x_{data}+t\epsilon$, velocity $\epsilon-x_{data}$, sampling $t:1\to0$.
Подставьте $s=1-t$. Ниже проверяем состояния и знак скорости; это не утверждение об одинаковом scheduler всех моделей.


In [ ]:
data=torch.tensor([2.,-1.],dtype=torch.float64)
noise=torch.tensor([-1.,1.5],dtype=torch.float64)
s=torch.linspace(0,1,41,dtype=torch.float64)[:,None];t=1-s
mit_path=(1-s)*noise+s*data
industrial_path=(1-t)*data+t*noise
assert torch.allclose(mit_path,industrial_path)
assert torch.allclose(data-noise,-(noise-data))
x=noise.clone()
for _ in range(20):x+=(-1/20)*(noise-data)
assert torch.allclose(x,data)
results['time_reversal_max_error']=float((mit_path-industrial_path).abs().max())
plt.figure(figsize=(5,3));plt.plot(*mit_path.T,'o-',markersize=3)
plt.scatter(*noise,label='noise');plt.scatter(*data,label='data')
plt.legend();plt.title('Один путь, две параметризации времени');plt.tight_layout();plt.show()


## 3. Conditional endpoint не равен marginal predictor

Пусть чистый endpoint принимает значения $z\in\{-2,2\}$ с probabilities .3/.7.
При фиксированном t: $p_t(x\mid z)=\mathcal N(tz,(1-t)^2)$.
Conditional velocity $u(x,t\mid z)=(z-x)/(1-t)$ известна. Sampler не знает z;
ему нужно $u(x,t)=\sum_z p(z\mid x_t=x)u(x,t\mid z)$.

Posterior weights зависят от x. Проверим также Gaussian-path relation:
$u=x/t+(1-t)s/t$ для interior t. Формула не предназначена для t=0 или 1 без анализа пределов.


In [ ]:
time=0.6;beta=1-time
endpoints=torch.tensor([-2.,2.],dtype=torch.float64)
prior=torch.tensor([.3,.7],dtype=torch.float64)
x=torch.linspace(-3,3,201,dtype=torch.float64).requires_grad_(True)
log_components=-.5*((x[:,None]-time*endpoints)/beta)**2+prior.log()
posterior=log_components.softmax(-1)
conditional_velocity=(endpoints-x[:,None])/beta
marginal_velocity=(posterior*conditional_velocity).sum(-1)
wrong_prior_average=(prior*conditional_velocity).sum(-1)
log_marginal=torch.logsumexp(log_components,dim=-1)-np.log(beta*np.sqrt(2*np.pi))
score=torch.autograd.grad(log_marginal.sum(),x)[0]
conversion=x/time+beta/time*score
error=float((marginal_velocity-conversion).abs().max().detach())
assert error<1e-10
results['marginal_velocity_score_conversion_error']=error
fig,axes=plt.subplots(1,2,figsize=(9,3))
axes[0].plot(x.detach(),posterior.detach());axes[0].set_title('Posterior weights зависят от x')
axes[1].plot(x.detach(),marginal_velocity.detach(),label='posterior average')
axes[1].plot(x.detach(),wrong_prior_average.detach(),'--',label='prior average: неверно')
axes[1].legend();axes[1].set_title('Поле sampler');plt.tight_layout();plt.show()


**Связь с обучением:** MSE regression на endpoint-specific targets имеет оптимум в виде conditional expectation.
В реальном Lab 2 marginal field аппроксимирует сеть. Этот пример вычисляет его точно и не выполняет training.

## 4. CFG: две ветви — не две разные архитектуры

Для одной сети, обученной с condition dropout:
$u_w=u_\varnothing+w(u_y-u_\varnothing)$.
Проверим w=0/1 и equivalence batched/separate evaluation на маленькой случайной MLP.
Здесь **не обучалось условное изображение**: проверяются tensor contracts и affine combination.


In [ ]:
net=torch.nn.Sequential(torch.nn.Linear(5,16),torch.nn.SiLU(),torch.nn.Linear(16,2)).eval()
x=torch.randn(8,2);time=torch.ones(8,1)*.4
null=torch.zeros(8,2);condition=torch.ones(8,2)
with torch.no_grad():
    u0=net(torch.cat([x,time,null],-1));uy=net(torch.cat([x,time,condition],-1))
    both=net(torch.cat([torch.cat([x,time,null],-1),torch.cat([x,time,condition],-1)],0))
assert torch.allclose(both[:8],u0) and torch.allclose(both[8:],uy)
assert torch.allclose(u0+0*(uy-u0),u0)
assert torch.allclose(u0+1*(uy-u0),uy)
print('Two separate batch-8 calls = one batch-16 call; branch work remains two samples per original sample.')
results['cfg_identities']='passed; random network, no conditional generation claim'


## 5. Patchify и стоимость tokens

Чистая перестановка pixels в patches обратима. Learned projection в hidden dimension сама по себе не обязана быть обратимой.
Проверим перестановку через unfold/fold. Затем **только арифметически** посчитаем размер naive attention scores;
большой tensor не выделяется. Flash/SDPA kernels могут не материализовывать всю эту матрицу.


In [ ]:
image=torch.arange(2*3*8*12,dtype=torch.float32).reshape(2,3,8,12)
patches=torch.nn.functional.unfold(image,kernel_size=2,stride=2).transpose(1,2)
restored=torch.nn.functional.fold(patches.transpose(1,2),output_size=(8,12),kernel_size=2,stride=2)
assert torch.equal(image,restored)
print('Image:',tuple(image.shape),'Patches:',tuple(patches.shape))
# Illustrative codec t4/s8 + patch (1,2,2), NOT a universal shape rule.
for frames,height,width in [(33,240,432),(121,480,864)]:
    assert (frames-1)%4==0 and height%16==0 and width%16==0
    latent_t=1+(frames-1)//4
    n=latent_t*(height//16)*(width//16)
    naive_bytes=1*16*n*n*2  # B=1, heads=16, fp16 scores only
    print((frames,height,width),'tokens=',n,'naive scores GiB=',round(naive_bytes/2**30,2))
results['patch_roundtrip']='exact; no learned projection involved'


## 6. VAE: std, KL и заморозка

Если encoder выдаёт logvar, $z=\mu+\exp(\tfrac12\,logvar)\epsilon$.
Для diagonal Gaussian KL к standard normal:
$\frac12\sum_j(\mu_j^2+\exp(logvar_j)-1-logvar_j)$.
Сначала суммируем latent dimensions, затем усредняем batch; другое reduction меняет относительный вес KL.

Во второй части замораживаем toy encoder/decoder и проверяем gradient только у latent predictor.
Это проверка wiring будущего LDM, **не обучение настоящего VAE/DiT**.


In [ ]:
mu=torch.zeros(4,6);logvar=torch.zeros_like(mu)
kl=.5*(mu.square()+logvar.exp()-1-logvar).sum(-1).mean()
assert kl.item()==0
std=(.5*torch.tensor(np.log(4.))).exp()
assert torch.allclose(std,torch.tensor(2.,dtype=std.dtype))
codec=torch.nn.ModuleDict({'encoder':torch.nn.Linear(8,4),'decoder':torch.nn.Linear(4,8)}).requires_grad_(False)
predictor=torch.nn.Linear(4,4)
before={k:v.detach().clone() for k,v in codec.state_dict().items()}
with torch.no_grad():latent=codec['encoder'](torch.randn(16,8))
loss=(predictor(latent)-torch.randn_like(latent)).square().mean();loss.backward()
assert all(p.grad is None for p in codec.parameters())
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in predictor.parameters())
assert all(torch.equal(v,before[k]) for k,v in codec.state_dict().items())
results['frozen_codec_gradients']='none; predictor gradients finite'
print('KL(q=N(0,I) || N(0,I)) =',kl.item(),'; frozen codec wiring passed')


## 7. Optional MLX parity — после PyTorch correctness

По умолчанию пропущено; не устанавливает MLX и не скачивает weights.
Если позже создадите отдельное MLX окружение с Torch, передавайте одинаковые массивы,
а не просто одинаковый seed. Здесь только Linear+MSE, не порт K6.
Latency сравнивайте отдельно после warm-up с принудительной evaluation/synchronization.


In [ ]:
RUN_MLX=False
if RUN_MLX:
    import mlx.core as mx
    xx=rng.normal(size=(8,4)).astype(np.float32)
    ww=rng.normal(size=(3,4)).astype(np.float32);bb=np.zeros(3,dtype=np.float32)
    pt_w=torch.tensor(ww,requires_grad=True);pt_b=torch.tensor(bb,requires_grad=True)
    pt_y=torch.tensor(xx)@pt_w.T+pt_b;pt_loss=pt_y.square().mean()
    pt_grads=torch.autograd.grad(pt_loss,[pt_w,pt_b])
    params={'w':mx.array(ww),'b':mx.array(bb)}
    def objective(p):return mx.mean((mx.array(xx)@p['w'].T+p['b'])**2)
    mx_loss,mx_grads=mx.value_and_grad(objective)(params);mx.eval(mx_loss,mx_grads)
    error=max(float(np.max(np.abs(np.array(mx_grads[k])-g.detach().numpy()))) for k,g in zip(['w','b'],pt_grads))
    assert error<1e-5
    results['mlx']={'linear_mse_gradient_max_error':error,'scope':'tiny Linear only'}
else:
    print('MLX skipped. CPU educational checks remain usable without it.')
(OUT/'probability-and-contracts-results.json').write_text(json.dumps(results,indent=2)+chr(10))
print(json.dumps(results,indent=2))


## Что делать дальше

- Перед Lab 1: объяснить batch vs dimensions и Gaussian score.
- Перед Lab 2: вывести reverse-time sign, conditional/marginal field и область определения conversion.
- Перед Lab 3: проверить CFG, patch shapes, logvar и frozen-stage optimizer.
- Для ComfyUI: выполнить [C0/C4](../COMFYUI_LABS.md), сохранив те же assumptions и честные ограничения измерений.
- Для реальной модели: читать [архитектурные точки входа](../INDUSTRY_BRIDGE.md), а не переносить toy dimensions автоматически.

Эти numerical checks не заменяют самостоятельную реализацию и обучение исходных Lab 1–3.
